In [9]:
#SOFTINT AI — Credit Scoring Model Training Pipeline
#Loads CSV → Encodes features → Trains 2 models → Evaluates → Saves best model
#"""#

import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix, classification_report



In [10]:
# ── Configuration ──
DATA_PATH = Path("data/synthetic/credit_scoring_dataset.csv")
TEST_DATA_PATH = Path("data/synthetic/test_set_1000.csv")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

TARGET = "default_next_12m"
DROP_COLS = ["customer_id"]


In [11]:
# ── Feature Types ──
NUMERIC = [
    "annual_salary_gbp", "credit_card_utilization_pct", "on_time_payment_ratio_pct",
    "rent_payment_monthly_gbp", "rent_on_time_rate_pct", "utility_bills_on_time_pct",
    "mobile_money_tx_monthly", "mobile_money_age_days", "preferred_loan_term_months",
    "existing_loans_count", "months_credit_history"
]
CATEGORICAL = ["gender", "occupation", "mortgage_status", "mobile_money_active"]

# ── Load Data ──
def load_data(path):
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}\n→ Run the generator script first!")
    df = pd.read_csv(path)
    print(f"✅ Loaded {len(df):,} profiles from {path.name}")
    return df


In [12]:
# ── Build Preprocessor ──
def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUMERIC),
            ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), CATEGORICAL)
        ])


In [13]:
# ── Train & Evaluate ──
def train_and_evaluate(X_train, X_test, y_train, y_test, model, name):
    print(f"\n{'='*50}")
    print(f"🤖 MODEL: {name}")
    print(f"{'='*50}")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    roc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)

    print(f"📊 Accuracy:  {acc:.1%}")
    print(f"📊 Precision: {prec:.1%}")
    print(f"📊 Recall:    {rec:.1%}")
    print(f"📊 F1 Score:  {f1:.1%}")
    print(f"📊 AUC-ROC:   {roc:.3f}")
    print(f"\n📋 Classification Report:\n{classification_report(y_test, y_pred, zero_division=0)}")
    print(f"🔍 Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

    return model, roc

# ── Fairness Check ──
def fairness_check(df_test, preds):
    print(f"\n{'='*50}")
    print(f"⚖️  FAIRNESS CHECK — Default Rate by Gender")
    print(f"{'='*50}")
    df_test["predicted"] = preds
    for g in sorted(df_test["gender"].unique()):
        subset = df_test[df_test["gender"] == g]
        rate = subset["predicted"].mean()
        actual = subset[TARGET].mean()
        print(f"   {g:10} Predicted Default: {rate:.1%}  |  Actual: {actual:.1%}")
    print(f"   ✅ Gap between groups should be <5% for fairness compliance")

# ── Predict New Applicant Example ──
def score_new_applicant(pipeline, applicant):
    """Score a single new applicant dictionary"""
    df = pd.DataFrame([applicant])
    proba = pipeline.predict_proba(df)[:, 1][0]
    score = int(850 - (proba * 550))  # Map 0-1 → 300-850 scale
    grade = "A" if score>=750 else "B" if score>=650 else "C" if score>=550 else "D" if score>=400 else "E"
    return {"score_300_850": score, "grade": grade, "default_risk_pct": round(proba*100,1)}

# ── MAIN EXECUTION ──
if __name__ == "__main__":
    print("🚀 SOFTINT AI — Credit Scoring Model Training Pipeline\n")

    # Load main dataset
    df = load_data(DATA_PATH)
    X = df.drop([TARGET]+DROP_COLS, axis=1)
    y = df[TARGET]

    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    df_test = df.loc[X_test.index].copy()
    print(f"📊 Train: {len(X_train):,} | Test: {len(X_test):,} | Default Rate: {y.mean():.1%}")

    # Preprocessor
    pre = build_preprocessor()

    # Model 1 — Logistic Regression
    lr_pipe = Pipeline(steps=[("pre", pre), ("model", LogisticRegression(max_iter=1000))])
    lr_model, lr_roc = train_and_evaluate(X_train, X_test, y_train, y_test, lr_pipe, "Logistic Regression")

    # Model 2 — Random Forest
    rf_pipe = Pipeline(steps=[("pre", pre), ("model", RandomForestClassifier(n_estimators=100, random_state=42))])
    rf_model, rf_roc = train_and_evaluate(X_train, X_test, y_train, y_test, rf_pipe, "Random Forest")

    # Fairness on best model
    best_model = rf_model if rf_roc >= lr_roc else lr_model
    fairness_check(df_test, best_model.predict(X_test))

    # Save best model
    model_path = MODEL_DIR / "credit_scoring_model.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(best_model, f)
    print(f"\n💾 Best model saved → {model_path}")

    # ── Example: Score a new applicant ──
    print(f"\n{'='*50}")
    print(f"🧪 EXAMPLE — Score New Applicant")
    print(f"{'='*50}")
    applicant = {
        "gender": "Female",
        "occupation": "Professional",
        "annual_salary_gbp": 58000,
        "credit_card_utilization_pct": 28.5,
        "on_time_payment_ratio_pct": 96.0,
        "mortgage_status": "Current",
        "rent_payment_monthly_gbp": 0,
        "rent_on_time_rate_pct": 100.0,
        "utility_bills_on_time_pct": 98.0,
        "mobile_money_active": True,
        "mobile_money_tx_monthly": 15,
        "mobile_money_age_days": 720,
        "preferred_loan_term_months": 24,
        "existing_loans_count": 1,
        "months_credit_history": 48
    }
    result = score_new_applicant(best_model, applicant)
    print(f"   Credit Score: {result['score_300_850']} / 850")
    print(f"   Grade:        {result['grade']}")
    print(f"   Risk:         {result['default_risk_pct']}% probability of default")

🚀 SOFTINT AI — Credit Scoring Model Training Pipeline

✅ Loaded 50,000 profiles from credit_scoring_dataset.csv
📊 Train: 40,000 | Test: 10,000 | Default Rate: 19.5%

🤖 MODEL: Logistic Regression
📊 Accuracy:  80.4%
📊 Precision: 46.7%
📊 Recall:    6.2%
📊 F1 Score:  10.9%
📊 AUC-ROC:   0.719

📋 Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.98      0.89      8054
           1       0.47      0.06      0.11      1946

    accuracy                           0.80     10000
   macro avg       0.64      0.52      0.50     10000
weighted avg       0.75      0.80      0.74     10000

🔍 Confusion Matrix:
[[7917  137]
 [1826  120]]

🤖 MODEL: Random Forest
📊 Accuracy:  80.2%
📊 Precision: 40.4%
📊 Recall:    3.4%
📊 F1 Score:  6.3%
📊 AUC-ROC:   0.697

📋 Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.99      0.89      8054
           1       0.40      0.03      0.06      1946

    